In [41]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")    

print("Using device:", device)

Using device: mps


1. Load Dataset
2. Basic Preprocessing
3. Training Process
    a. create the model
    b. Forward pass
    c. Loss calculation
    d. Back Propagation
    e. Parameters updates
4. Model evaluation

In [42]:
import torch
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
torch.manual_seed(42)

In [43]:
url = "https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/master/data.csv"
df = pd.read_csv(url)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [44]:
print(df.head())

         id diagnosis  ...  fractal_dimension_worst  Unnamed: 32
0    842302         M  ...                  0.11890          NaN
1    842517         M  ...                  0.08902          NaN
2  84300903         M  ...                  0.08758          NaN
3  84348301         M  ...                  0.17300          NaN
4  84358402         M  ...                  0.07678          NaN

[5 rows x 33 columns]


In [45]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)


In [46]:
print(df.shape)

(569, 31)


In [47]:
print(df['diagnosis'].value_counts())

diagnosis
B    357
M    212
Name: count, dtype: int64


In [48]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

In [49]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,stratify=y, random_state=42)

In [50]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [51]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)


In [52]:
X_train_tensor = torch.from_numpy(X_train).to(dtype = torch.float32, device = device)
X_test_tensor = torch.from_numpy(X_test).to(dtype = torch.float32, device = device)  
y_train_tensor = torch.from_numpy(y_train).to(dtype = torch.float32, device = device)
y_test_tensor = torch.from_numpy(y_test).to(dtype = torch.float32, device = device)

In [53]:
print(X_train_tensor.shape, X_test_tensor.shape, y_train_tensor.shape, y_test_tensor.shape)

torch.Size([455, 30]) torch.Size([114, 30]) torch.Size([455]) torch.Size([114])


##### Defining the Model

In [54]:
class BreastCancerModel():

    def __init__(self,X):

      self.weights = torch.rand(X_train_tensor.shape[1],1, device = device, dtype = torch.float32, requires_grad=True)
      self.bias  = torch.rand(1, device = device, dtype = torch.float32, requires_grad=True )

    def forward_pass (self,X):
        z = torch.matmul(X,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self,y_pred,y):
        loss = torch.nn.functional.binary_cross_entropy(y_pred,y)
        return loss
    


In [55]:
learning_rate = 0.01
epocs = 25

In [56]:
#Training pipeline
model = BreastCancerModel(X_train_tensor)
print(model.weights.shape, model.bias.shape)

# define loop 
for epoch in range(epocs):
    # forward pass
    y_pred = model.forward_pass(X_train_tensor).squeeze(1) # squeeze to remove the extra dimension(455,1) to (455,)
    # loss calculation
    loss = model.loss_function(y_pred,y_train_tensor)
    
    # backward pass
    loss.backward()

    #parameter update
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # zero the gradients 
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    print(f"Epoch:{epoch+1}, Loss:{loss.item():.4f}")

torch.Size([30, 1]) torch.Size([1])
Epoch:1, Loss:0.8296
Epoch:2, Loss:0.8289
Epoch:3, Loss:0.8281
Epoch:4, Loss:0.8273
Epoch:5, Loss:0.8266
Epoch:6, Loss:0.8259
Epoch:7, Loss:0.8252
Epoch:8, Loss:0.8244
Epoch:9, Loss:0.8237
Epoch:10, Loss:0.8230
Epoch:11, Loss:0.8222
Epoch:12, Loss:0.8215
Epoch:13, Loss:0.8207
Epoch:14, Loss:0.8201
Epoch:15, Loss:0.8193
Epoch:16, Loss:0.8186
Epoch:17, Loss:0.8179
Epoch:18, Loss:0.8172
Epoch:19, Loss:0.8164
Epoch:20, Loss:0.8157
Epoch:21, Loss:0.8150
Epoch:22, Loss:0.8143
Epoch:23, Loss:0.8135
Epoch:24, Loss:0.8129
Epoch:25, Loss:0.8121


In [57]:
print(model.weights, model.bias)

tensor([[ 8.2546e-01],
        [ 5.9289e-01],
        [ 5.6509e-01],
        [ 7.7663e-01],
        [ 3.3609e-01],
        [ 4.4269e-01],
        [ 6.6711e-01],
        [-3.2152e-04],
        [ 3.9907e-01],
        [ 1.5748e-01],
        [ 5.9932e-01],
        [ 8.3925e-01],
        [ 4.6598e-01],
        [ 2.1041e-01],
        [ 4.1084e-01],
        [ 7.0324e-01],
        [ 7.7677e-01],
        [ 2.3346e-01],
        [ 9.1650e-01],
        [ 3.2310e-01],
        [ 9.5952e-01],
        [ 6.8306e-01],
        [ 8.0170e-01],
        [ 5.8963e-01],
        [ 3.1485e-02],
        [ 7.7290e-01],
        [ 8.5294e-01],
        [ 8.9263e-01],
        [ 7.6217e-01],
        [ 7.5107e-01]], device='mps:0', requires_grad=True) tensor([0.0612], device='mps:0', requires_grad=True)


In [71]:
# model evaluation
with torch.no_grad():
    y_pred_test = model.forward_pass(X_test_tensor).squeeze(1)
    y_pred_test_class = (y_pred_test >= 0.65).float()
    accuracy = accuracy_score(y_test_tensor.cpu(), y_pred_test_class.cpu())
    print(f"Test Accuracy: {accuracy:.4f}")
    print(classification_report(y_test_tensor.cpu(), y_pred_test_class.cpu()))

Test Accuracy: 0.9123
              precision    recall  f1-score   support

         0.0       0.93      0.93      0.93        72
         1.0       0.88      0.88      0.88        42

    accuracy                           0.91       114
   macro avg       0.91      0.91      0.91       114
weighted avg       0.91      0.91      0.91       114

